# Imports and Setup


In [19]:
import pandas as pd
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

version = "1"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [20]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

### Check Fiscal Year


In [21]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
current_fy_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [22]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in current_fy_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

2027
FY27

2026
FY26


In [23]:
str_sum_current_adopted_amount = f"{current_fy_name}_current_adopted_amount"
str_sum_current_modified_amount = f"{current_fy_name}_current_modified_amount"
str_sum_financial_plan_amount = f"{planned_fy_name}_financial_plan_amount"

str_sum_current_adopted_position = f"{current_fy_name}_current_adopted_position"
str_sum_current_modified_position = f"{current_fy_name}_current_modified_position"
str_sum_financial_plan_position = f"{planned_fy_name}_financial_plan_position"

select_arr = [
    "publication_date",
    "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "budget_code_number",
    "budget_code_name",
    "financial_plan_savings_flag",
    f"SUM(adopted_budget_amount) AS {str_sum_current_adopted_amount}",
    f"SUM(current_modified_budget_amount) AS {str_sum_current_modified_amount}",
    f"SUM(financial_plan_amount) AS {str_sum_financial_plan_amount}",
    f"SUM(adopted_budget_position) AS {str_sum_current_adopted_position}",
    f"SUM(current_modified_budget_position) AS {str_sum_current_modified_position}",
    f"SUM(financial_plan_position) AS {str_sum_financial_plan_position}"
]

select_string = ", ".join(select_arr)

print(f"select_string:\n{select_string}\n")

where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")


groupby_arr = [
    "publication_date",
    "fiscal_year",
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "budget_code_number",
    "budget_code_name",
    "financial_plan_savings_flag"
]

groupby_string = ", ".join(groupby_arr)


print(f"groupby_string:\n{groupby_string}\n")



orderby_arr = [
    "agency_number",
    "unit_appropriation_number",
    "budget_code_number",
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")



limit = 1000

select_string:
publication_date, fiscal_year, agency_number, agency_name, unit_appropriation_number, unit_appropriation_name, budget_code_number, budget_code_name, financial_plan_savings_flag, SUM(adopted_budget_amount) AS FY26_current_adopted_amount, SUM(current_modified_budget_amount) AS FY26_current_modified_amount, SUM(financial_plan_amount) AS FY27_financial_plan_amount, SUM(adopted_budget_position) AS FY26_current_adopted_position, SUM(current_modified_budget_position) AS FY26_current_modified_position, SUM(financial_plan_position) AS FY27_financial_plan_position

where_string:
fiscal_year IN (2027)

groupby_string:
publication_date, fiscal_year, agency_number, agency_name, unit_appropriation_number, unit_appropriation_name, budget_code_number, budget_code_name, financial_plan_savings_flag

orderby_string:
agency_number, unit_appropriation_number, budget_code_number, publication_date DESC



### Query Data if necessary


In [ ]:
# query_string = f"SELECT {select_string} WHERE {where_string} GROUP BY {groupby_string} ORDER BY {orderby_string} LIMIT {limit}"
# print(query_string)

data = []
offset = 0


# max_item = 0
try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}_v{version}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            select=select_string,
            where=where_string,
            group=groupby_string,
            order=orderby_string,
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv", index=False)    

In [25]:
df.columns

Index(['publication_date', 'fiscal_year', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'budget_code_number', 'budget_code_name', 'financial_plan_savings_flag',
       'FY26_current_adopted_amount', 'FY26_current_modified_amount',
       'FY27_financial_plan_amount', 'FY26_current_adopted_position',
       'FY26_current_modified_position', 'FY27_financial_plan_position'],
      dtype='str')

### Convert to ints


In [26]:


for col in [
    str_sum_current_adopted_amount,
    str_sum_current_modified_amount,
    str_sum_financial_plan_amount,
    str_sum_current_adopted_position,
    str_sum_current_modified_position,
    str_sum_financial_plan_position,
]:
    print(df[col].dtype)
    df[col] = df[col].astype(int)
    print(df[col].dtype)
    print()

int64
int64

int64
int64

int64
int64

int64
int64

int64
int64

int64
int64



In [27]:
df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,financial_plan_savings_flag,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position
0,20260512,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N,14934531,14697531,14937263,149,109,109
1,20260217,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N,14934531,14934531,14937263,149,109,109
2,20260512,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N,351980,351980,351980,2,2,2
3,20260217,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N,351980,351980,351980,2,2,2
4,20260512,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0214,First Deputy Mayor,N,2306539,2157539,2306539,14,14,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16212,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,P010,Fleet Reduction,Y,0,0,-1060000000,0,0,0
16213,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,0,0,-400000000,0,0,0
16214,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,0,0,37689973,0,0,0
16215,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,Tier IV RECs,Y,0,0,75000000,0,0,0


## Track releases


In [28]:
pub_dates = df["publication_date"].sort_values(ascending=True).unique().tolist()

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

2 pub_dates in FY 2027: [20260217, 20260512]


## Set up base DataFrame and map function


In [29]:
base_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "budget_code_number",
    "budget_code_name",
    "financial_plan_savings_flag",
]

base_df = df[base_cols].drop_duplicates()
base_df.head()

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,financial_plan_savings_flag
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N
2,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N
4,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0214,First Deputy Mayor,N
6,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0217,D/M FOR HEALTH & HUMAN SERVICES,N
8,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0218,SPECIAL EVENTS,N


In [30]:
AGENCY_MAP = df[["agency_number","agency_name"]].drop_duplicates().set_index('agency_number')['agency_name'].to_dict()

UOA_MAP = df.groupby('agency_number')[['unit_appropriation_number', 'unit_appropriation_name']].apply(
    lambda x: dict(zip(x['unit_appropriation_number'], x['unit_appropriation_name']))
).to_dict()

BUDGETCODE_MAP = df.groupby(['agency_number', 'unit_appropriation_number'])[["budget_code_number","budget_code_name"]].apply(lambda x: dict(zip(x['budget_code_number'], x['budget_code_name']))).to_dict()

BUDGETCODE_MAP

{('002', 20): {'0211': 'CHIEF OF STAFF',
  '0213': 'Office of ThriveNYC',
  '0214': 'First Deputy Mayor',
  '0217': 'D/M FOR HEALTH & HUMAN SERVICES',
  '0218': 'SPECIAL EVENTS',
  '0220': 'Intergovernmental Affairs',
  '0222': 'Deputy Mayor for Strategic Policy',
  '0225': 'D/M ECONOMIC DEVEL',
  '0226': 'D/M for Housing & Economic Development',
  '0229': 'Counsel to the Mayor',
  '0230': "Mayor's Judiciary Committee",
  '0235': 'D/M FOR OPERATIONS',
  '0243': 'Citywide Capital Services',
  '0245': 'Comm to Combat Domestic Violence',
  '0248': 'Public Design Commission',
  '0250': 'Office of Immigrant Affairs',
  '0253': 'CAPITAL PROJECT DEVELOPMENT',
  '0264': 'NYC Service Office',
  '0274': 'Citywide Events Coordination & Mgmt.',
  '0277': 'Senior Advisor to the Mayor',
  '0298': 'RECORDS MANAGEMENT GRANT',
  '0332': 'NYC Tourism Grant',
  'P020': 'FINANCIAL PLAN SAVINGS'},
 ('002', 21): {'0211': 'CHIEF OF STAFF',
  '0214': 'First Deputy Mayor',
  '0217': 'D/M FOR HEALTH & HUMAN SER

## Loop through each release


In [31]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

[20260217, 20260512]
2027
0 prelim
1 exec


## Budget Codes


In [32]:
target_cols = [
    "agency_number",
    "unit_appropriation_number",
    "budget_code_number",
    "financial_plan_savings_flag",
    f"{str_sum_current_adopted_amount}",
    f"{str_sum_current_modified_amount}",
    f"{str_sum_financial_plan_amount}",
    f"{str_sum_current_adopted_position}",
    f"{str_sum_current_modified_position}",
    f"{str_sum_financial_plan_position}",
]


collapsed_df = base_df.copy()

for i, pub_date in enumerate(pub_dates):
    ith_release_df = df[df["publication_date"] == pub_date][target_cols].copy()
    
    print(i)
    
    if i <= 2:
        ith_release_df = ith_release_df.rename(
            columns={
                f"{str_sum_financial_plan_amount}":f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}",
                f"{str_sum_financial_plan_position}":f"{str_sum_financial_plan_position}_{RELEASE_NAMES[i]}"
                }
        )
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    if i < num_pubs_this_year - 1:
        ith_release_df = ith_release_df.drop(columns=[
            f"{str_sum_current_adopted_amount}",
            f"{str_sum_current_modified_amount}",
            f"{str_sum_current_adopted_position}",
            f"{str_sum_current_modified_position}"
            ])
    
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=["agency_number","unit_appropriation_number","budget_code_number", "financial_plan_savings_flag"], how='left')

collapsed_df = collapsed_df.fillna(0)

0
[0/2] pub_date -- 20260217 (prelim):
Index(['agency_number', 'unit_appropriation_number', 'budget_code_number',
       'financial_plan_savings_flag', 'FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim'],
      dtype='str')

1
[1/2] pub_date -- 20260512 (exec):
Index(['agency_number', 'unit_appropriation_number', 'budget_code_number',
       'financial_plan_savings_flag', 'FY26_current_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_current_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec'],
      dtype='str')



In [33]:
print(collapsed_df.columns)

collapsed_df.head()

Index(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'budget_code_number', 'budget_code_name',
       'financial_plan_savings_flag', 'FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim', 'FY26_current_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_current_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec'],
      dtype='str')


,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,financial_plan_savings_flag,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N,14937263.0,109.0,14934531.0,14697531.0,14937263.0,149.0,109.0,109.0
1,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N,351980.0,2.0,351980.0,351980.0,351980.0,2.0,2.0,2.0
2,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0214,First Deputy Mayor,N,2306539.0,14.0,2306539.0,2157539.0,2306539.0,14.0,14.0,14.0
3,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0217,D/M FOR HEALTH & HUMAN SERVICES,N,1348827.0,7.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0
4,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0218,SPECIAL EVENTS,N,1989369.0,20.0,1989369.0,1989369.0,1989369.0,20.0,20.0,20.0


In [34]:
# for j in range(0,i):
#     print(f"i={i}, j={j}")
#     print(f"{RELEASE_NAMES[i]} - {RELEASE_NAMES[j]}")
    
#     i_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[i]}"
#     j_amount_name = f"{str_sum_financial_plan_amount}_{RELEASE_NAMES[j]}"
    
#     i_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[i]}"
#     j_position_name = f"{str_sum_financial_plan_position}_{RELEASE_NAMES[j]}"
    
#     print()
#     print(f"i_amount_name: {i_amount_name}")
#     print(f"j_amount_name: {j_amount_name}")
#     print()
#     print(f"i_position_name: {i_position_name}")
#     print(f"j_position_name: {j_position_name}")
#     print()
        
        
#     amount_change_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
#     amount_change_percent_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
    
#     print(amount_change_name)
#     print(amount_change_percent_name)
        
#     collapsed_df[amount_change_name] = collapsed_df[i_amount_name] - collapsed_df[j_amount_name]
#     collapsed_df[amount_change_percent_name] = (collapsed_df[i_amount_name] - collapsed_df[j_amount_name])/collapsed_df[i_amount_name]
    
    
#     print()
    
#     position_change_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
#     position_change_percent_name = f"{str_sum_financial_plan_position}_%change_{RELEASE_NAMES[i]}-{RELEASE_NAMES[j]}"
    
#     print(position_change_name)
#     print(position_change_percent_name)
    
#     collapsed_df[position_change_name] = collapsed_df[i_position_name] - collapsed_df[j_position_name]
#     collapsed_df[position_change_percent_name] = (collapsed_df[i_position_name] - collapsed_df[j_position_name])/collapsed_df[i_position_name]
    
#     print("="*50)

In [35]:
collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,financial_plan_savings_flag,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N,1.493726e+07,109.0,14934531.0,14697531.0,14937263.0,149.0,109.0,109.0
1,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N,3.519800e+05,2.0,351980.0,351980.0,351980.0,2.0,2.0,2.0
2,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0214,First Deputy Mayor,N,2.306539e+06,14.0,2306539.0,2157539.0,2306539.0,14.0,14.0,14.0
3,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0217,D/M FOR HEALTH & HUMAN SERVICES,N,1.348827e+06,7.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0
4,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0218,SPECIAL EVENTS,N,1.989369e+06,20.0,1989369.0,1989369.0,1989369.0,20.0,20.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8255,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,P010,Fleet Reduction,Y,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8256,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,0.000000e+00,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0
8257,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,3.768997e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8258,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,Tier IV RECs,Y,7.500000e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
# # "total_adopted_budget_amount",
# # "total_current_modified_budget_amount",
# # "total_adopted_budget_position",
# # "total_current_modified_budget_position"

# str_sum_current_adopted_amount
# str_sum_current_modified_amount

# str_sum_current_adopted_position
# str_sum_current_modified_position


# amount_change_prev_adopt_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
# amount_change_percent_prev_adopt_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"

# collapsed_df[amount_change_prev_adopt_name]= collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount]
# collapsed_df[amount_change_percent_prev_adopt_name]= (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_adopted_amount])/collapsed_df[i_amount_name]


# amount_change_prev_modified_name = f"{str_sum_financial_plan_amount}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"
# amount_change_percent_prev_modified_name = f"{str_sum_financial_plan_amount}_%change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

# collapsed_df[amount_change_prev_modified_name] = collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount]
# collapsed_df[amount_change_percent_prev_modified_name] = (collapsed_df[i_amount_name] - collapsed_df[str_sum_current_modified_amount])/collapsed_df[i_amount_name]


# position_change_prev_adopt_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_adopt"
# position_change_prev_modified_name = f"{str_sum_financial_plan_position}_change_{RELEASE_NAMES[i]}-{current_fy_name}_modified"

# collapsed_df[position_change_prev_adopt_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_adopted_position]
# collapsed_df[position_change_prev_modified_name] = collapsed_df[i_position_name] - collapsed_df[str_sum_current_modified_position]

# # collapsed_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-adopt"] = (collapsed_df[i_position_name] - collapsed_df["total_adopted_budget_position"])/collapsed_df[i_position_name]
# # collapsed_df[f"total_financial_plan_position_%change_{RELEASE_NAMES[i]}-modified"] = (collapsed_df[i_position_name] - collapsed_df["total_current_modified_budget_position"])/collapsed_df[i_position_name]

In [37]:
collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,financial_plan_savings_flag,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0211,CHIEF OF STAFF,N,1.493726e+07,109.0,14934531.0,14697531.0,14937263.0,149.0,109.0,109.0
1,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0213,Office of ThriveNYC,N,3.519800e+05,2.0,351980.0,351980.0,351980.0,2.0,2.0,2.0
2,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0214,First Deputy Mayor,N,2.306539e+06,14.0,2306539.0,2157539.0,2306539.0,14.0,14.0,14.0
3,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0217,D/M FOR HEALTH & HUMAN SERVICES,N,1.348827e+06,7.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0
4,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0218,SPECIAL EVENTS,N,1.989369e+06,20.0,1989369.0,1989369.0,1989369.0,20.0,20.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8255,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,P010,Fleet Reduction,Y,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8256,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,0.000000e+00,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0
8257,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,Y,3.768997e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8258,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,Tier IV RECs,Y,7.500000e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
for c in collapsed_df.columns:
    print(c)

agency_number
agency_name
unit_appropriation_number
unit_appropriation_name
budget_code_number
budget_code_name
financial_plan_savings_flag
FY27_financial_plan_amount_prelim
FY27_financial_plan_position_prelim
FY26_current_adopted_amount
FY26_current_modified_amount
FY27_financial_plan_amount_exec
FY26_current_adopted_position
FY26_current_modified_position
FY27_financial_plan_position_exec


In [39]:
collapsed_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Budget Codes",index=False)

## Units of Appropriation


In [40]:
collapsed_df.columns

Index(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'budget_code_number', 'budget_code_name',
       'financial_plan_savings_flag', 'FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim', 'FY26_current_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_current_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec'],
      dtype='str')

In [41]:
print(len(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'financial_plan_savings_flag']))


5


In [42]:
UoA_collapsed_df = collapsed_df.groupby(['agency_number', 'agency_name', 'unit_appropriation_number', 'unit_appropriation_name', 'financial_plan_savings_flag']).sum().reset_index()

# print(UoA_collapsed_df.columns[7:])

target_cols = UoA_collapsed_df.columns[7:]

UoA_collapsed_df = UoA_collapsed_df[['agency_number', 'agency_name', 'unit_appropriation_number', 'unit_appropriation_name', 'financial_plan_savings_flag',*target_cols]]

UoA_collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,financial_plan_savings_flag,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,N,4.046508e+07,278.0,40515979.0,40316729.0,40465085.0,319.0,279.0,278.0
1,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,Y,0.000000e+00,40.0,0.0,0.0,6286117.0,0.0,40.0,44.0
2,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,N,4.489982e+06,0.0,4237982.0,4459520.0,5031799.0,0.0,0.0,0.0
3,002,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS,N,5.588977e+07,467.0,55263233.0,54555072.0,56564894.0,455.0,452.0,474.0
4,002,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS,Y,1.774270e+05,0.0,177416.0,177416.0,-664573.0,0.0,0.0,-7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
970,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,Y,-2.637000e+03,0.0,22363.0,97.0,-2637.0,0.0,0.0,0.0
971,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,Y,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0
972,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,Y,0.000000e+00,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0
973,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,Y,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
UoA_collapsed_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="UoA",index=False)

## Agencies


In [44]:
UoA_collapsed_df.columns

Index(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'financial_plan_savings_flag',
       'FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim', 'FY26_current_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_current_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec'],
      dtype='str')

In [45]:
Agency_collapsed_df = UoA_collapsed_df.groupby(['agency_number', 'agency_name']).sum().reset_index()

print(Agency_collapsed_df.columns[5:])

target_cols = Agency_collapsed_df.columns[5:]

Agency_collapsed_df = Agency_collapsed_df[['agency_number', 'agency_name', *target_cols]]

Agency_collapsed_df

Index(['FY27_financial_plan_amount_prelim',
       'FY27_financial_plan_position_prelim', 'FY26_current_adopted_amount',
       'FY26_current_modified_amount', 'FY27_financial_plan_amount_exec',
       'FY26_current_adopted_position', 'FY26_current_modified_position',
       'FY27_financial_plan_position_exec'],
      dtype='str')


,agency_number,agency_name,FY27_financial_plan_amount_prelim,FY27_financial_plan_position_prelim,FY26_current_adopted_amount,FY26_current_modified_amount,FY27_financial_plan_amount_exec,FY26_current_adopted_position,FY26_current_modified_position,FY27_financial_plan_position_exec
0,002,MAYORALTY,1.987918e+08,1402.0,206176143.0,207283515.0,209965286.0,1374.0,1397.0,1423.0
1,003,BOARD OF ELECTIONS,1.468751e+08,517.0,146875148.0,216647571.0,206972679.0,517.0,517.0,517.0
2,004,CAMPAIGN FINANCE BOARD,1.347705e+07,87.0,109481237.0,123881237.0,104093091.0,258.0,258.0,258.0
3,008,OFFICE OF THE ACTUARY,7.617044e+06,42.0,7614099.0,7677099.0,8483139.0,42.0,42.0,42.0
4,010,BOROUGH PRESIDENT - MANHATTAN,5.586632e+06,56.0,6017382.0,6194656.0,6079019.0,56.0,56.0,56.0
...,...,...,...,...,...,...,...,...,...,...
140,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,6.555800e+05,5.0,673602.0,673602.0,657628.0,5.0,5.0,5.0
141,99C,CITYWIDE SAVINGS INITIATIVES,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0
142,99D,PRIOR YEAR PAYABLES,0.000000e+00,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0
143,99P,ENERGY ADJUSTMENT,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
UoA_collapsed_df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="UoA",index=False)